Challenge, add up amount spent by cx ID, print id for a customer and the total of all their orders from 'customer-orders.csv' in descending order
Rows are CustomerID, OrderID, Amount

In [1]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
# In order to create a schema, need to create a structtype object, array of struct field objects


spark = SparkSession.builder.appName('CxAmounts').getOrCreate()
spark.sparkContext.setLogLevel("ERROR")


schema = StructType([

    #Column name, type of variable/input, whether it can be null or not
    StructField('CxID', StringType(), True),
    StructField('OrderID', StringType(), True),
    StructField('Amount', FloatType(), True)
])

df = spark.read.csv(
    '../customer-orders.csv',
    schema=schema,
    header=False
)

df.printSchema()

# Want to find total customer orders
totalOrders = df.groupBy('CxID').agg(func.round(func.sum('Amount'), 2).alias('totalAmount')) 
totalOrders = totalOrders.sort('totalAmount', ascending=False) # Sorts by totalAmount in descending order
totalOrders.show()


spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/06 14:50:22 WARN Utils: Your hostname, CartersDell, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/06 14:50:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/06 14:50:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- CxID: string (nullable = true)
 |-- OrderID: string (nullable = true)
 |-- Amount: float (nullable = true)



+----+-----------+
|CxID|totalAmount|
+----+-----------+
|  68|    6375.45|
|  73|     6206.2|
|  39|    6193.11|
|  54|    6065.39|
|  71|    5995.66|
|   2|    5994.59|
|  97|    5977.19|
|  46|    5963.11|
|  42|    5696.84|
|  59|    5642.89|
|  41|    5637.62|
|   0|    5524.95|
|   8|    5517.24|
|  85|    5503.43|
|  61|    5497.48|
|  32|    5496.05|
|  58|    5437.73|
|  63|    5415.15|
|  15|    5413.51|
|   6|    5397.88|
+----+-----------+
only showing top 20 rows


Doing the same as above but with sql query:
Challenge: add up amount spent by cx ID, print id for a customer and the total of all their orders from 'customer-orders.csv' in descending order
Rows in file are CustomerID, OrderID, Amount

In [2]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
# In order to create a schema, need to create a structtype object, array of struct field objects


spark = SparkSession.builder.appName('CxAmounts').getOrCreate()
# spark.sparkContext.setLogLevel("ERROR")


schema = StructType([

    #Column name, type of variable/input, whether it can be null or not
    StructField('CxID', StringType(), True),
    StructField('OrderID', StringType(), True),
    StructField('Amount', FloatType(), True)
])

df = spark.read.csv(
    '../customer-orders.csv',
    schema=schema,
    header=False
)

# df.printSchema()

# Want to find total customer orders
# totalOrders = df.groupBy('CxID').agg(func.round(func.sum('Amount'), 2).alias('totalAmount')) 
# totalOrders = totalOrders.sort('totalAmount', ascending=False) # Sorts by totalAmount in descending order
# totalOrders.show()

df.createOrReplaceTempView('Orders')

customer = spark.sql("""
    SELECT CxID, round(SUM(Amount), 2) as CxTotal
    FROM Orders
    GROUP BY CxID
    ORDER BY CxTotal DESC      
""")

customer.show()
spark.stop()

+----+-------+
|CxID|CxTotal|
+----+-------+
|  68|6375.45|
|  73| 6206.2|
|  39|6193.11|
|  54|6065.39|
|  71|5995.66|
|   2|5994.59|
|  97|5977.19|
|  46|5963.11|
|  42|5696.84|
|  59|5642.89|
|  41|5637.62|
|   0|5524.95|
|   8|5517.24|
|  85|5503.43|
|  61|5497.48|
|  32|5496.05|
|  58|5437.73|
|  63|5415.15|
|  15|5413.51|
|   6|5397.88|
+----+-------+
only showing top 20 rows
